# GraphDict and MemoryGraphList Tutorial

This tutorial covers the core data structures in `kgcnn_torch` for representing graphs:

- **GraphDict**: A dictionary-based container for storing graph tensors (numpy arrays)
- **MemoryGraphList**: A list of `GraphDict` objects, representing a dataset of graphs
- **Graph Preprocessors**: Operations for manipulating graph topology (adding self-loops, making edges undirected, etc.)
- **PyG Conversion**: Converting `MemoryGraphList` to PyTorch Geometric `Data` objects via `to_pyg_list()`

These classes mirror the Keras KGCNN `GraphDict` and `MemoryGraphList` but are designed to work with PyTorch and PyG.

## 1. Creating a GraphDict from NetworkX

A `GraphDict` behaves like a Python dictionary whose values are numpy arrays. We can build one from a NetworkX graph.

In [ ]:
import networkx as nx
import numpy as np

# Create the Karate Club graph
G = nx.karate_club_graph()
nx.draw(G, with_labels=True, node_size=300)

In [ ]:
from kgcnn_torch.graph.base import GraphDict

# Create an empty GraphDict and populate from NetworkX
graph_dict = GraphDict()
graph_dict.from_networkx(G, node_attributes="club")
print("Keys:", list(graph_dict.keys()))
print("Club labels:", graph_dict["club"][:5])

## 2. GraphDict: Set, Get, and Manipulate Properties

`GraphDict` provides `assign_property()` (aliased as `set()`) and `obtain_property()` (aliased as `get()`) for safely handling property assignment. Unlike raw dict access, `set()` ignores `None` values and automatically casts to numpy arrays.

In [ ]:
# Create a GraphDict from scratch
g = GraphDict({
    "edge_indices": np.array([[0, 1], [1, 0], [1, 2], [2, 1]]),
    "node_attributes": np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
})
print("Graph:", g)

In [ ]:
# assign_property / set: adds a property, auto-casts lists to numpy arrays
g.set("graph_labels", [0])  # list -> np.array
g.set("edge_weights", [[1.0], [1.0], [0.5], [0.5]])
print("graph_labels:", g["graph_labels"], type(g["graph_labels"]))
print("edge_weights:", g["edge_weights"])

In [ ]:
# set() with None does nothing (safe assignment)
g.set("missing_property", None)
print("'missing_property' in g:", "missing_property" in g)

In [ ]:
# obtain_property / get: returns None if key not found (no KeyError)
print("node_attributes:", g.obtain_property("node_attributes").shape)
print("nonexistent:", g.obtain_property("nonexistent"))

In [ ]:
# search_properties: find properties matching a regex pattern
print("All 'edge_*' properties:", g.search_properties("^edge_.*"))
print("All 'node_*' properties:", g.search_properties("^node_.*"))

In [ ]:
# has_valid_key / assert_has_valid_key
print("Has 'edge_indices':", g.has_valid_key("edge_indices"))
print("Has 'missing':", g.has_valid_key("missing"))

In [ ]:
# copy() makes a shallow copy
g_copy = g.copy()
g_copy.set("extra", [42])
print("Original has 'extra':", "extra" in g)
print("Copy has 'extra':", "extra" in g_copy)

In [ ]:
# Convert back to NetworkX
G_back = g.to_networkx()
print("NetworkX graph nodes:", list(G_back.nodes()))
print("NetworkX graph edges:", list(G_back.edges()))

## 3. Graph Preprocessors

Preprocessors transform graph properties (e.g., add self-loops, make edges undirected, normalize weights). In `kgcnn_torch`, preprocessors are standalone classes rather than methods on `GraphDict`. Each preprocessor takes a `GraphDict` and returns a new `GraphDict` with the transformed properties.

You can apply them in two ways:
1. Call the preprocessor directly and `update()` the graph
2. Use `graph_dict.apply_preprocessor(name, **kwargs)` with a string name

In [ ]:
from kgcnn_torch.graph.preprocessor import (
    AddEdgeSelfLoops, MakeUndirectedEdges, SortEdgeIndices,
    NormalizeEdgeWeightsSymmetric, SetEdgeWeightsUniform,
    CountNodesAndEdges
)

In [ ]:
# Start with the Karate Club graph
graph_dict = GraphDict()
graph_dict.from_networkx(G, node_attributes="club")
print("Before preprocessing:")
print("  Num edges:", len(graph_dict["edge_indices"]))

# Method 1: Apply preprocessors directly
graph_dict.update(AddEdgeSelfLoops()(graph_dict))
graph_dict.update(MakeUndirectedEdges()(graph_dict))
graph_dict.update(SortEdgeIndices()(graph_dict))
graph_dict.update(NormalizeEdgeWeightsSymmetric()(graph_dict))

print("\nAfter preprocessing:")
print("  Num edges:", len(graph_dict["edge_indices"]))
print("  Edge weights shape:", graph_dict["edge_weights"].shape)
print("  Edge weights sample:", graph_dict["edge_weights"][:3].flatten())

In [ ]:
# Method 2: Use apply_preprocessor with string name
g2 = GraphDict({
    "edge_indices": np.array([[0, 1], [1, 2]]),
    "node_attributes": np.array([[1.0], [2.0], [3.0]])
})
g2.apply_preprocessor("add_edge_self_loops")
g2.apply_preprocessor("make_undirected_edges")
g2.apply_preprocessor("sort_edge_indices")
print("Edges after apply_preprocessor:", g2["edge_indices"])

In [ ]:
# Inspect preprocessor configuration
print(AddEdgeSelfLoops().get_config())
print(NormalizeEdgeWeightsSymmetric().get_config())

In [ ]:
# in_place=True applies changes directly to the input GraphDict
from kgcnn_torch.graph.preprocessor import SetAngle

SetAngle(
    range_indices="edge_indices",
    angle_indices="angle_combinations",
    in_place=True
)(graph_dict)

print("Angle indices shape:", graph_dict["angle_combinations"].shape)
print("Angle node triples shape:", graph_dict["angle_indices_nodes"].shape)

### Important Warning

If you change graph tensors (redefine edges, delete nodes), dependent tensors (attributes, angles, index maps) are **not** automatically updated, since `GraphDict` is a plain dictionary of tensor values. You must re-run the relevant preprocessors.

## 4. MemoryGraphList: Working with Graph Datasets

`MemoryGraphList` is a list of `GraphDict` objects. It provides batch operations like `assign_property()`, `obtain_property()`, `map_list()`, `tensor()`, and `clean()`.

In [ ]:
from kgcnn_torch.data.base import MemoryGraphList

# Create a list of 3 deterministic toy graphs
graphs = MemoryGraphList([
    GraphDict({
        "node_number": np.array([6, 6, 8]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.2, 0.0, 0.0], [0.0, 1.1, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 0], [1, 2], [2, 1]]),
        "graph_labels": np.array([1.5], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([7, 6]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.1, 0.2, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 0]]),
        "graph_labels": np.array([2.3], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([6, 8, 7, 6]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.1, 0.0, 0.0], [2.0, 0.3, 0.0], [0.5, 1.2, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 0], [1, 2], [2, 1], [0, 3], [3, 0]]),
        "graph_labels": np.array([0.8], dtype=np.float32),
    }),
])
print("Number of graphs:", len(graphs))
print(graphs)


In [ ]:
# assign_property: set a property across all graphs
graphs.assign_property("dataset_id", [np.array([0]), np.array([1]), np.array([2])])

# obtain_property: get a property from all graphs as a list
labels = graphs.obtain_property("graph_labels")
print("Labels:", labels)

In [ ]:
# Indexing: int returns GraphDict, slice/list returns MemoryGraphList
print("Type of graphs[0]:", type(graphs[0]))
print("Type of graphs[:2]:", type(graphs[:2]))
print("Type of graphs[[0, 2]]:", type(graphs[[0, 2]]))

In [ ]:
# map_list: apply a preprocessor to every graph in the list
graphs.map_list(method="add_edge_self_loops")
graphs.map_list(method="sort_edge_indices")
graphs.map_list(method="count_nodes_and_edges")

print("Graph 0 total_nodes:", graphs[0]["total_nodes"])
print("Graph 0 total_edges:", graphs[0]["total_edges"])

In [ ]:
# tensor(): convert graph properties to batched numpy arrays (padded)
inputs = [
    {"shape": [None, 1], "name": "node_number", "dtype": "int64"},
    {"shape": [None, 2], "name": "edge_indices", "dtype": "int64"},
]
tensors = graphs.tensor(inputs)
print("Padded node_number shape:", tensors[0].shape)  # (3, max_nodes)
print("Padded edge_indices shape:", tensors[1].shape)  # (3, max_edges, 2)

In [ ]:
# clean(): remove graphs that have missing or empty required properties
graphs_with_bad = MemoryGraphList([
    GraphDict({"node_number": np.array([6, 8]), "edge_indices": np.array([[0, 1], [1, 0]])}),
    GraphDict({"node_number": np.array([6]), "edge_indices": np.array([])}),  # empty edges
    GraphDict({"node_number": np.array([7, 6]), "edge_indices": np.array([[0, 1]])}),
])
removed = graphs_with_bad.clean(["edge_indices"])
print("Removed indices:", removed)
print("Remaining graphs:", len(graphs_with_bad))

In [ ]:
# rename_property_on_graphs
graphs.rename_property_on_graphs("dataset_id", "sample_id")
print("Renamed: 'sample_id' in graphs[0]:", "sample_id" in graphs[0])
print("Renamed: 'dataset_id' in graphs[0]:", "dataset_id" in graphs[0])

In [ ]:
# save / load MemoryGraphList to pickle
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "graphs.pkl")
    graphs.save(path)
    loaded = MemoryGraphList()
    loaded.load(path)
    print("Loaded", len(loaded), "graphs")
    print("Loaded graph[0] keys:", list(loaded[0].keys()))

## 5. Converting to PyG Data Objects

`MemoryGraphList.to_pyg_list()` converts the entire graph list to a list of PyTorch Geometric `Data` objects. This is the primary interface for feeding data into PyTorch GNN models.

**Important**: The conversion automatically swaps the edge index convention from KGCNN `(target, source)` to PyG `(source, target)`.

In [ ]:
import torch
from torch_geometric.data import Data

pyg_list = graphs.to_pyg_list()

print("Number of PyG Data objects:", len(pyg_list))
print("\nFirst graph:")
print(pyg_list[0])
print("  z (node features):", pyg_list[0].z)
print("  edge_index shape:", pyg_list[0].edge_index.shape)
print("  y (labels):", pyg_list[0].y)

In [ ]:
# Use with PyG DataLoader
from torch_geometric.loader import DataLoader

loader = DataLoader(pyg_list, batch_size=2, shuffle=False)

for batch in loader:
    print("Batch:")
    print("  x:", batch.x.shape if batch.x is not None else "None")
    print("  z:", batch.z.shape if hasattr(batch, 'z') and batch.z is not None else "None")
    print("  edge_index:", batch.edge_index.shape)
    print("  batch:", batch.batch)
    print("  y:", batch.y)
    break

## 6. Full Example: Build, Preprocess, and Convert

Here is a complete workflow: create graphs, apply preprocessors, and convert to PyG format ready for training.

In [ ]:
# Build a small molecular-like dataset with deterministic toy graphs
dataset = MemoryGraphList([
    GraphDict({
        "node_number": np.array([6, 6, 8]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.2, 0.0, 0.0], [0.0, 1.1, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 2]]),
        "graph_labels": np.array([0.12], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([6, 7, 8, 1]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.3, 0.1, 0.0], [2.2, 0.0, 0.0], [1.1, 0.9, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 2], [1, 3]]),
        "graph_labels": np.array([-0.05], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([8, 6, 6]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.0, 0.8, 0.0], [1.9, 0.1, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 2], [0, 2]]),
        "graph_labels": np.array([0.33], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([7, 6, 6, 8, 1]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [2.0, 0.0, 0.0], [1.0, 1.0, 0.0], [2.5, 0.7, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 2], [1, 3], [2, 4]]),
        "graph_labels": np.array([-0.18], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([6, 6, 6, 1]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.4, 0.0, 0.0], [2.8, 0.0, 0.0], [1.4, 1.0, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 2], [1, 3]]),
        "graph_labels": np.array([0.07], dtype=np.float32),
    }),
    GraphDict({
        "node_number": np.array([8, 6, 7, 1]),
        "node_coordinates": np.array([[0.0, 0.0, 0.0], [1.1, 0.0, 0.0], [2.1, 0.3, 0.0], [0.8, 1.0, 0.0]], dtype=np.float32),
        "edge_indices": np.array([[0, 1], [1, 2], [0, 3]]),
        "graph_labels": np.array([0.29], dtype=np.float32),
    }),
])

print(f"Created {len(dataset)} graphs")


In [ ]:
# Preprocess: make undirected, add self-loops, compute ranges
dataset.map_list(method="make_undirected_edges")
dataset.map_list(method="add_edge_self_loops")
dataset.map_list(method="sort_edge_indices")
dataset.map_list(method="set_edge_weights_uniform")
dataset.map_list(method="normalize_edge_weights_sym")

print("Graph 0 edge_indices shape:", dataset[0]["edge_indices"].shape)
print("Graph 0 edge_weights shape:", dataset[0]["edge_weights"].shape)

In [ ]:
# Convert to PyG and create DataLoader
pyg_data = dataset.to_pyg_list()
loader = DataLoader(pyg_data, batch_size=16, shuffle=True)

batch = next(iter(loader))
print("Batch object:", batch)
print("Number of graphs in batch:", batch.num_graphs)
print("Total nodes:", batch.x.shape[0] if batch.x is not None else "(see z)")

## Summary

| Class | Purpose |
|---|---|
| `GraphDict` | Dict-like container for a single graph's tensors (numpy arrays) |
| `MemoryGraphList` | List of `GraphDict`, with batch operations (`map_list`, `tensor`, `clean`, `to_pyg_list`) |
| `MemoryGraphDataset` | Extends `MemoryGraphList` with file I/O, CSV reading, and dataset management |
| Preprocessors (`AddEdgeSelfLoops`, etc.) | Standalone graph transformation classes |
| `to_pyg_list()` | Convert to PyG `Data` objects (swaps edge index convention automatically) |

The typical workflow is:
1. Create/load `MemoryGraphList` with graph data
2. Apply preprocessors via `map_list()`
3. Convert to PyG via `to_pyg_list()`
4. Feed into `DataLoader` for training